# Dataset 5: laboratory-result, specialty, and medication EDA

This split contains glucose and A1C result categories, demographics, specialty, and medication-status variables.

The notebook analyses this split independently. The original CSV is not changed; the cleaned copy is saved in `EDA_Dataset` for later joining on `encounter_id`.

## Setup

In [1]:
from pathlib import Path  # Import required library

import matplotlib.pyplot as plt  # Import required library
import pandas as pd  # Import required library
import seaborn as sns  # Import required library

sns.set_theme(style="whitegrid", context="notebook")  # Set the chart theme
pd.set_option("display.max_rows", 100)  # Configure dataframe display
pd.set_option("display.max_columns", 50)  # Configure dataframe display

In [2]:
def show_category_counts(series, title, top_n=15):  # Define the categorical-frequency chart helper
    counts = series.fillna("Missing").value_counts().head(top_n).sort_values()  # Calculate category frequencies
    fig, ax = plt.subplots(figsize=(10, max(3.5, 0.38 * len(counts))))  # Create the chart area
    ax.barh(counts.index.astype(str), counts.values, color="#4C78A8")  # Draw the horizontal bar chart
    ax.set_title(f"{title}: top {top_n} categories")  # Label the chart axis or title
    ax.set_xlabel("Encounter count")  # Label the chart axis or title
    ax.set_ylabel("")  # Label the chart axis or title
    for y, value in enumerate(counts.values):  # Add value labels to each bar
        ax.text(value, y, f" {value:,}", va="center", fontsize=8)  # Run this EDA or data-cleaning step
    plt.tight_layout()  # Adjust chart spacing
    plt.show()  # Display the chart


def show_numeric_distribution(series, title, discrete=False):  # Define the numeric-distribution chart helper
    values = series.dropna()  # Remove missing numeric values before plotting
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))  # Create the chart area
    sns.histplot(values, discrete=discrete, ax=axes[0], color="#4C78A8")  # Plot the feature distribution
    axes[0].set_title(f"{title}: distribution")  # Configure the numeric chart
    axes[0].set_xlabel(title)  # Configure the numeric chart
    sns.boxplot(x=values, ax=axes[1], color="#72B7B2")  # Plot the feature box plot
    axes[1].set_title(f"{title}: box plot")  # Configure the numeric chart
    axes[1].set_xlabel(title)  # Configure the numeric chart
    plt.tight_layout()  # Adjust chart spacing
    plt.show()  # Display the chart

## Load the source data

In [3]:
source_path = Path("divided_datasets") / "dataset_5.csv"  # Set the path to the source CSV
df_raw = pd.read_csv(source_path, na_values=["?"], keep_default_na=True, low_memory=False)  # Load the source CSV and convert question marks to missing values

## Initial inspection

In [4]:
df_raw.head()  # Inspect the original dataframe

,encounter_id,max_glu_serum,gender,tolazamide,miglitol,metformin,citoglipton,medical_specialty,A1Cresult
0,2278392,NaN,Female,No,No,No,No,Pediatrics-Endocrinology,NaN
1,149190,NaN,Female,No,No,No,No,NaN,NaN
2,64410,NaN,Female,No,No,No,No,NaN,NaN
3,500364,NaN,Male,No,No,No,No,NaN,NaN
4,16680,NaN,Male,No,No,No,No,NaN,NaN


In [5]:
df_raw.info()  # Inspect the original dataframe

<class 'pandas.DataFrame'>
RangeIndex: 101766 entries, 0 to 101765
Data columns (total 9 columns):
 #   Column             Non-Null Count   Dtype
---  ------             --------------   -----
 0   encounter_id       101766 non-null  int64
 1   max_glu_serum      5346 non-null    str  
 2   gender             101766 non-null  str  
 3   tolazamide         101766 non-null  str  
 4   miglitol           101766 non-null  str  
 5   metformin          101766 non-null  str  
 6   citoglipton        101766 non-null  str  
 7   medical_specialty  51817 non-null   str  
 8   A1Cresult          17018 non-null   str  
dtypes: int64(1), str(8)
memory usage: 9.2 MB


In [6]:
df_raw.shape  # Inspect the original dataframe

(101766, 9)

In [7]:
df_raw.describe()  # Inspect the original dataframe

,encounter_id
count,1.017660e+05
mean,1.652016e+08
std,1.026403e+08
min,1.252200e+04
25%,8.496119e+07
50%,1.523890e+08
75%,2.302709e+08
max,4.438672e+08


In [8]:
df_raw.isna().sum().sort_values(ascending=False).to_frame(name="missing_count")  # Inspect the original dataframe

,missing_count
max_glu_serum,96420
A1Cresult,84748
medical_specialty,49949
encounter_id,0
gender,0
miglitol,0
tolazamide,0
citoglipton,0
metformin,0


In [9]:
df_raw.duplicated().sum()  # Inspect the original dataframe

np.int64(0)

In [10]:
df_raw["encounter_id"].duplicated().sum()  # Run this EDA or data-cleaning step

np.int64(0)

In [11]:
df_raw["encounter_id"].is_unique  # Run this EDA or data-cleaning step

True

In [12]:
df_raw.nunique(dropna=False).sort_values().to_frame(name="unique_values_including_missing")  # Inspect the original dataframe

,unique_values_including_missing
citoglipton,1
tolazamide,3
gender,3
max_glu_serum,4
metformin,4
miglitol,4
A1Cresult,4
medical_specialty,73
encounter_id,101766


## Source-field checks before cleaning

### Maximum serum glucose result before missing-value handling

In [13]:
df_raw["max_glu_serum"].value_counts(dropna=False).head(10).to_frame(name="count")  # Run this EDA or data-cleaning step

,count
max_glu_serum,
NaN,96420
Norm,2597
>200,1485
>300,1264


In [14]:
show_category_counts(df_raw["max_glu_serum"], "Maximum serum glucose result before missing-value handling", top_n=10)  # Run this EDA or data-cleaning step

C:\Users\anind\AppData\Local\Temp\ipykernel_10728\4155684497.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()  # Display the chart


### A1C result before missing-value handling

In [15]:
df_raw["A1Cresult"].value_counts(dropna=False).head(10).to_frame(name="count")  # Run this EDA or data-cleaning step

,count
A1Cresult,
NaN,84748
>8,8216
Norm,4990
>7,3812


In [16]:
show_category_counts(df_raw["A1Cresult"], "A1C result before missing-value handling", top_n=10)  # Run this EDA or data-cleaning step

C:\Users\anind\AppData\Local\Temp\ipykernel_10728\4155684497.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()  # Display the chart


## Cleaning decisions

- Missing-value normalization: Question-mark and blank text values are treated as missing.
- Lab-result categories: Missing max_glu_serum and A1Cresult values are marked with boolean flags and labeled Not measured; this preserves the fact that a test result was absent.
- Medical specialty: A missingness flag is retained and absent values are labeled Unknown for visualization.
- Gender: Unknown/Invalid is normalized to Unknown.
- Constant feature removed: citoglipton contains one observed value (No) and adds no analytical variation.
- Rows and join key: No rows are removed. encounter_id remains unchanged and unique for the later join.

In [17]:
df = df_raw.copy()  # Copy the source dataframe before cleaning

text_columns = df.select_dtypes(include=["object", "string"]).columns  # Identify text columns for normalization
for column in text_columns:  # Clean each text column
    df[column] = df[column].astype("string").str.strip().replace("", pd.NA)  # Strip whitespace and convert blank text to missing values

df["max_glu_serum_missing"] = df["max_glu_serum"].isna()  # Create or handle glucose-result missingness information
df["max_glu_serum"] = df["max_glu_serum"].fillna("Not measured")  # Create or handle glucose-result missingness information

df["A1Cresult_missing"] = df["A1Cresult"].isna()  # Create or handle A1C-result missingness information
df["A1Cresult"] = df["A1Cresult"].fillna("Not measured")  # Create or handle A1C-result missingness information

df["medical_specialty_missing"] = df["medical_specialty"].isna()  # Create or handle specialty missingness information
df["medical_specialty"] = df["medical_specialty"].fillna("Unknown")  # Create or handle specialty missingness information

df["gender"] = df["gender"].replace({"Unknown/Invalid": "Unknown"})  # Normalize the gender category

assert df["citoglipton"].nunique(dropna=False) == 1, "citoglipton is expected to be constant"  # Validate the cleaning result
columns_removed = ["citoglipton"]  # Record columns removed during cleaning
df = df.drop(columns=columns_removed)  # Remove the documented uninformative columns

assert len(df) == len(df_raw), "No encounter rows should be removed"  # Validate the cleaning result
assert df["encounter_id"].notna().all(), "The join key must not be missing"  # Validate the cleaning result
assert df["encounter_id"].is_unique, "The join key must remain unique"  # Validate the cleaning result
assert df.columns.is_unique, "Duplicate column names are not allowed"  # Validate the cleaning result

In [18]:
cleaning_summary = pd.DataFrame({  # Create the cleaning-decision summary
    "decision": ["Missing-value normalization","Lab-result categories","Medical specialty","Gender","Constant feature removed","Rows and join key","Columns removed"],  # Add a cleaning-summary field
    "detail": ["Question-mark and blank text values are treated as missing.","Missing max_glu_serum and A1Cresult values are marked with boolean flags and labeled Not measured; this preserves the fact that a test result was absent.","A missingness flag is retained and absent values are labeled Unknown for visualization.","Unknown/Invalid is normalized to Unknown.","citoglipton contains one observed value (No) and adds no analytical variation.","No rows are removed. encounter_id remains unchanged and unique for the later join.","Columns removed after review: citoglipton"]  # Add a cleaning-summary field
})  # Run this EDA or data-cleaning step
cleaning_summary  # Create the cleaning-decision summary

,decision,detail
0,Missing-value normalization,Question-mark and blank text values are treate...
1,Lab-result categories,Missing max_glu_serum and A1Cresult values are...
2,Medical specialty,A missingness flag is retained and absent valu...
3,Gender,Unknown/Invalid is normalized to Unknown.
4,Constant feature removed,citoglipton contains one observed value (No) a...
5,Rows and join key,No rows are removed. encounter_id remains unch...
6,Columns removed,Columns removed after review: citoglipton


## Post-cleaning validation

In [19]:
df.head()  # Run this EDA or data-cleaning step

,encounter_id,max_glu_serum,gender,tolazamide,miglitol,metformin,medical_specialty,A1Cresult,max_glu_serum_missing,A1Cresult_missing,medical_specialty_missing
0,2278392,Not measured,Female,No,No,No,Pediatrics-Endocrinology,Not measured,True,True,False
1,149190,Not measured,Female,No,No,No,Unknown,Not measured,True,True,True
2,64410,Not measured,Female,No,No,No,Unknown,Not measured,True,True,True
3,500364,Not measured,Male,No,No,No,Unknown,Not measured,True,True,True
4,16680,Not measured,Male,No,No,No,Unknown,Not measured,True,True,True


In [20]:
df.info()  # Run this EDA or data-cleaning step

<class 'pandas.DataFrame'>
RangeIndex: 101766 entries, 0 to 101765
Data columns (total 11 columns):
 #   Column                     Non-Null Count   Dtype 
---  ------                     --------------   ----- 
 0   encounter_id               101766 non-null  int64 
 1   max_glu_serum              101766 non-null  string
 2   gender                     101766 non-null  string
 3   tolazamide                 101766 non-null  string
 4   miglitol                   101766 non-null  string
 5   metformin                  101766 non-null  string
 6   medical_specialty          101766 non-null  string
 7   A1Cresult                  101766 non-null  string
 8   max_glu_serum_missing      101766 non-null  bool  
 9   A1Cresult_missing          101766 non-null  bool  
 10  medical_specialty_missing  101766 non-null  bool  
dtypes: bool(3), int64(1), string(7)
memory usage: 10.9 MB


In [21]:
df.shape  # Run this EDA or data-cleaning step

(101766, 11)

In [22]:
df.describe()  # Run this EDA or data-cleaning step

,encounter_id
count,1.017660e+05
mean,1.652016e+08
std,1.026403e+08
min,1.252200e+04
25%,8.496119e+07
50%,1.523890e+08
75%,2.302709e+08
max,4.438672e+08


In [23]:
df.isna().sum().sort_values(ascending=False).to_frame(name="missing_count_after_cleaning")  # Run this EDA or data-cleaning step

,missing_count_after_cleaning
encounter_id,0
max_glu_serum,0
gender,0
tolazamide,0
miglitol,0
metformin,0
medical_specialty,0
A1Cresult,0
max_glu_serum_missing,0
A1Cresult_missing,0


In [24]:
df["encounter_id"].nunique()  # Inspect the selected cleaned column

101766

## Univariate EDA after cleaning

### Maximum serum glucose result

In [25]:
df["max_glu_serum"].value_counts(dropna=False).head(10).to_frame(name="count")  # Create or handle glucose-result missingness information

,count
max_glu_serum,
Not measured,96420
Norm,2597
>200,1485
>300,1264


In [26]:
show_category_counts(df["max_glu_serum"], "Maximum serum glucose result", top_n=10)  # Run this EDA or data-cleaning step

C:\Users\anind\AppData\Local\Temp\ipykernel_10728\4155684497.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()  # Display the chart


### Gender

In [27]:
df["gender"].value_counts(dropna=False).head(10).to_frame(name="count")  # Normalize the gender category

,count
gender,
Female,54708
Male,47055
Unknown,3


In [28]:
show_category_counts(df["gender"], "Gender", top_n=10)  # Run this EDA or data-cleaning step

C:\Users\anind\AppData\Local\Temp\ipykernel_10728\4155684497.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()  # Display the chart


### Tolazamide status

In [29]:
df["tolazamide"].value_counts(dropna=False).head(10).to_frame(name="count")  # Inspect the selected cleaned column

,count
tolazamide,
No,101727
Steady,38
Up,1


In [30]:
show_category_counts(df["tolazamide"], "Tolazamide status", top_n=10)  # Run this EDA or data-cleaning step

C:\Users\anind\AppData\Local\Temp\ipykernel_10728\4155684497.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()  # Display the chart


### Miglitol status

In [31]:
df["miglitol"].value_counts(dropna=False).head(10).to_frame(name="count")  # Inspect the selected cleaned column

,count
miglitol,
No,101728
Steady,31
Down,5
Up,2


In [32]:
show_category_counts(df["miglitol"], "Miglitol status", top_n=10)  # Run this EDA or data-cleaning step

C:\Users\anind\AppData\Local\Temp\ipykernel_10728\4155684497.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()  # Display the chart


### Metformin status

In [33]:
df["metformin"].value_counts(dropna=False).head(10).to_frame(name="count")  # Inspect the selected cleaned column

,count
metformin,
No,81778
Steady,18346
Up,1067
Down,575


In [34]:
show_category_counts(df["metformin"], "Metformin status", top_n=10)  # Run this EDA or data-cleaning step

C:\Users\anind\AppData\Local\Temp\ipykernel_10728\4155684497.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()  # Display the chart


### Medical specialty

In [35]:
df["medical_specialty"].value_counts(dropna=False).head(20).to_frame(name="count")  # Create or handle specialty missingness information

,count
medical_specialty,
Unknown,49949
InternalMedicine,14635
Emergency/Trauma,7565
Family/GeneralPractice,7440
Cardiology,5352
Surgery-General,3099
Nephrology,1613
Orthopedics,1400
Orthopedics-Reconstructive,1233


In [36]:
show_category_counts(df["medical_specialty"], "Medical specialty", top_n=20)  # Run this EDA or data-cleaning step

C:\Users\anind\AppData\Local\Temp\ipykernel_10728\4155684497.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()  # Display the chart


### A1C result

In [37]:
df["A1Cresult"].value_counts(dropna=False).head(10).to_frame(name="count")  # Create or handle A1C-result missingness information

,count
A1Cresult,
Not measured,84748
>8,8216
Norm,4990
>7,3812


In [38]:
show_category_counts(df["A1Cresult"], "A1C result", top_n=10)  # Run this EDA or data-cleaning step

C:\Users\anind\AppData\Local\Temp\ipykernel_10728\4155684497.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()  # Display the chart


## Save the cleaned split

In [39]:
output_dir = Path("EDA_Dataset")  # Set the cleaned-data output folder
output_dir.mkdir(exist_ok=True)  # Set the cleaned-data output folder
output_path = output_dir / "dataset_5.csv"  # Set the cleaned CSV output path
df.to_csv(output_path, index=False)  # Save the cleaned dataframe
print(f"Saved {df.shape[0]:,} rows and {df.shape[1]:,} columns to {output_path}")  # Report the saved dataset details

Saved 101,766 rows and 11 columns to EDA_Dataset\dataset_5.csv


In [40]:
saved_df = pd.read_csv(output_path)  # Reload the saved CSV for verification
assert saved_df.shape == df.shape, "Saved CSV shape does not match the cleaned dataframe"  # Validate the cleaning result
saved_df.head()  # Reload the saved CSV for verification

,encounter_id,max_glu_serum,gender,tolazamide,miglitol,metformin,medical_specialty,A1Cresult,max_glu_serum_missing,A1Cresult_missing,medical_specialty_missing
0,2278392,Not measured,Female,No,No,No,Pediatrics-Endocrinology,Not measured,True,True,False
1,149190,Not measured,Female,No,No,No,Unknown,Not measured,True,True,True
2,64410,Not measured,Female,No,No,No,Unknown,Not measured,True,True,True
3,500364,Not measured,Male,No,No,No,Unknown,Not measured,True,True,True
4,16680,Not measured,Male,No,No,No,Unknown,Not measured,True,True,True
